# 2. VQ-VAE Training (Stage 1)

**Objective:** Train the VQ-VAE model from `src.model.vae` on the full `P+C+S` sequences. We will run a small training loop here, plot the losses, and save the final model weights to `experiments/vqvae_stage1.pth`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!git clone https://github.com/irene-30/DLAI.git
%cd /content/DLAI

Cloning into 'DLAI'...
remote: Enumerating objects: 201, done.
remote: Counting objects: 100% (201/201), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 201 (delta 87), reused 129 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (201/201), 107.35 KiB | 4.47 MiB/s, done.
Resolving deltas: 100% (87/87), done.
/content/DLAI


In [ ]:
%pip install datasets transformers torch tqdm matplotlib

In [ ]:
import sys
import os
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm import tqdm
import matplotlib.pyplot as plt

# Add 'src' to path
#sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.utils import (
    get_llm_tokenizer, MAX_SEQ_LEN, PATH_VQVAE_MODEL,
    VQ_CODEBOOK_SIZE
)
from src.dataset import Lazy_VQVAE_Dataset
from src.model.vae import VQVAEModel

# --- Configuration ---
D_MODEL = 256
NUM_EPOCHS = 30 # Increase for a real run
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2.1 Load Tokenizer and Dataset

In [ ]:
tokenizer = get_llm_tokenizer()
vocab_size = len(tokenizer)

print(f"Tokenizer vocabulary size (including new tokens): {vocab_size}")

raw_dataset = load_dataset("gsm8k", "main")['train']
train_dataset = Lazy_VQVAE_Dataset(tokenizer, raw_dataset, max_length=MAX_SEQ_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Loaded {len(train_dataset)} samples for VQ-VAE training.")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Loaded 7473 samples for VQ-VAE training.


## 2.2 Initialize Model and Optimizer

In [ ]:
model = VQVAEModel(
    vocab_size=vocab_size,
    d_model=D_MODEL,
    num_embeddings=VQ_CODEBOOK_SIZE,
    max_seq_len=MAX_SEQ_LEN
).to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"VQ-VAE Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

VQ-VAE Model parameters: 30.52M


In [ ]:
import os
import torch.optim as optim

# --- 1. Definisci i percorsi per il Drive ---
drive_save_folder = "/content/drive/My Drive/TokenAssorted_GSM8K_Results/"
# Creiamo una cartella apposta per i checkpoint del VAE
vae_checkpoint_dir = os.path.join(drive_save_folder, "checkpoints", "vqvae")
os.makedirs(vae_checkpoint_dir, exist_ok=True)

# Questo è il file che useremo per salvare e caricare
CHECKPOINT_FILE = os.path.join(vae_checkpoint_dir, "vqvae_checkpoint2.pth")

# --- 2. Inizializza Modello e Ottimizzatore ---
model = VQVAEModel(
    vocab_size=vocab_size,
    d_model=D_MODEL,
    num_embeddings=VQ_CODEBOOK_SIZE,
    max_seq_len=MAX_SEQ_LEN
).to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"VQ-VAE Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

# --- 3. (NOVITÀ) Carica dal Checkpoint se esiste ---
start_epoch = 0
if os.path.exists(CHECKPOINT_FILE):
    print(f"ATTENZIONE: Trovato checkpoint esistente in {CHECKPOINT_FILE}")
    print("Caricamento del modello e dello stato dell'ottimizzatore...")

    checkpoint = torch.load(CHECKPOINT_FILE, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']

    print(f"Caricamento completato. Si riparte dall'epoca {start_epoch + 1}.")
else:
    print("Nessun checkpoint trovato. Inizio addestramento da zero.")

VQ-VAE Model parameters: 30.52M
Nessun checkpoint trovato. Inizio addestramento da zero.


## 2.3 Training Loop

We'll run the training loop directly in the notebook to monitor its progress and plot the losses.

In [ ]:
model.train()
losses = []
recon_losses = []
vq_losses = []

for epoch in range(start_epoch, NUM_EPOCHS):
    print(f"--- EPOCH {epoch+1}/{NUM_EPOCHS} ---")
    epoch_loss, epoch_recon, epoch_vq = 0, 0, 0

    for batch in tqdm(train_loader):
        input_ids = batch['input_ids'].to(device)

        optimizer.zero_grad()

        total_loss, recon_loss, vq_loss = model(input_ids)

        total_loss.backward()
        optimizer.step()

        epoch_loss += total_loss.item()
        epoch_recon += recon_loss.item()
        epoch_vq += vq_loss.item()

    # Log average losses for the epoch
    avg_loss = epoch_loss / len(train_loader)
    avg_recon = epoch_recon / len(train_loader)
    avg_vq = epoch_vq / len(train_loader)

    losses.append(avg_loss)
    recon_losses.append(avg_recon)
    vq_losses.append(avg_vq)

    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f} | Recon: {avg_recon:.4f} | VQ: {avg_vq:.4f}")
    print(f"Salvataggio checkpoint per l'epoca {epoch + 1}...")

    checkpoint = {
        'epoch': epoch + 1,  # Salva l'epoca *successiva* da cui ripartire
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_loss,
        'recon': avg_recon,
        'VQ loss': avg_vq
    }

    torch.save(checkpoint, CHECKPOINT_FILE)
    print(f"Checkpoint salvato in {CHECKPOINT_FILE}")

--- EPOCH 1/30 ---


100%|██████████| 468/468 [08:08<00:00,  1.04s/it]


Epoch 1 | Avg Loss: 2.5246 | Recon: 1.5157 | VQ: 1.0088
Salvataggio checkpoint per l'epoca 1...
Checkpoint salvato in /content/drive/My Drive/TokenAssorted_GSM8K_Results/checkpoints/vqvae/vqvae_checkpoint2.pth
--- EPOCH 2/30 ---


100%|██████████| 468/468 [08:13<00:00,  1.05s/it]


Epoch 2 | Avg Loss: 1.6681 | Recon: 0.8352 | VQ: 0.8329
Salvataggio checkpoint per l'epoca 2...
Checkpoint salvato in /content/drive/My Drive/TokenAssorted_GSM8K_Results/checkpoints/vqvae/vqvae_checkpoint2.pth
--- EPOCH 3/30 ---


...

100%|██████████| 468/468 [08:14<00:00,  1.06s/it]


Epoch 13 | Avg Loss: 0.7076 | Recon: 0.5732 | VQ: 0.1343
Salvataggio checkpoint per l'epoca 13...
Checkpoint salvato in /content/drive/My Drive/TokenAssorted_GSM8K_Results/checkpoints/vqvae/vqvae_checkpoint2.pth
--- EPOCH 14/30 ---


100%|██████████| 468/468 [08:15<00:00,  1.06s/it]


Epoch 14 | Avg Loss: 0.6788 | Recon: 0.5634 | VQ: 0.1155
Salvataggio checkpoint per l'epoca 14...
Checkpoint salvato in /content/drive/My Drive/TokenAssorted_GSM8K_Results/checkpoints/vqvae/vqvae_checkpoint2.pth
--- EPOCH 15/30 ---


 39%|███▉      | 183/468 [03:13<05:02,  1.06s/it]

## 2.4 Visualize Losses

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(losses, label='Total Loss')
plt.plot(recon_losses, label='Reconstruction Loss', linestyle='--')
plt.plot(vq_losses, label='VQ Loss', linestyle=':')
plt.title('VQ-VAE Training Losses')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

NameError: name 'plt' is not defined

## 2.5 Save the Model

Finally, we save the trained VQ-VAE weights. These will be loaded by the next notebook to create the assorted dataset.

In [ ]:
#print(f"Saving VQ-VAE model to {PATH_VQVAE_MODEL}")
#os.makedirs(os.path.dirname(PATH_VQVAE_MODEL), exist_ok=True)
#torch.save(model.state_dict(), PATH_VQVAE_MODEL)
#print("Model saved.")

Saving VQ-VAE model to experiments/vqvae_stage1.pth
Model saved.


In [ ]:
import os
drive_save_folder = "/content/drive/My Drive/DLAI/experiments/"

# 2. Assicurati che la cartella esista (la crea se non c'è)
os.makedirs(drive_save_folder, exist_ok=True)

# 3. Definisci il nome completo del file
final_model_path = os.path.join(drive_save_folder, "vqvae_stage(cc0.05).pth")

# 4. Salva il modello usando il percorso assoluto
print(f"Sto salvando il modello direttamente su Google Drive in:")
print(f"{final_model_path}")

torch.save(model.state_dict(), final_model_path)

print("--- Modello salvato con successo su Google Drive! ---")

Sto salvando il modello direttamente su Google Drive in:
/content/drive/My Drive/DLAI/experiments/vqvae_stage(cc0.1).pth
--- Modello salvato con successo su Google Drive! ---
